Preparations:
- load prediction.csv
    - drop the lines with the non-used ifs
    - if there is more then one if, take mean

- add column "sample" to master table
- join them
- drop unnecessary columns (matadata, paths, ranks)

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import ast
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, f1_score, precision_recall_curve, confusion_matrix

# Configuration
CSV_PATH =  "/home/bri/pymol/data/predictions/preds_snir.csv" 
MASTER_PATH =  "/home/bri/pymol/data/3ab/table/master_table_snir.csv" 
YAML_PATH = "/home/bri/pymol/EDA/static/metric_directions.yaml"
OUTPUT_DIR = Path("/home/bri/pymol/EDA/static/snir_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


KEEP_INTERFACES = {"A,C", "B,C"}   # as option if there is more than one, default A,B



# Load data
preds_df = pd.read_csv(CSV_PATH)
preds_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Columns: 138 entries, sample to esmfold_mpnn_nll_b
dtypes: bool(1), float64(127), str(10)
memory usage: 218.5 KB


## Filter and merge interfaces

In [14]:
df_if = preds_df[preds_df["interface"].str.strip().isin(KEEP_INTERFACES)].copy()
print(f"Loaded {len(df_if)} rows. Interfaces: {df_if['interface'].unique()}")

Loaded 66 rows. Interfaces: <ArrowStringArray>
['A,C', 'B,C']
Length: 2, dtype: str


In [15]:
# for snir data
master_df = pd.read_csv(MASTER_PATH, delimiter=';')

df_interface = pd.merge(master_df, df_if, on=['sample', 'interface'], how='inner')
# Display the first few rows to verify
print(df_interface[['sample', 'vh/vl', 'target', 'af3_plddt']].head())

# Save the merged result
df_interface.to_csv(OUTPUT_DIR /'merged_snir.csv', index=False)

         sample                                              vh/vl  \
0  5ifj_abc_cut  EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...   
1  5ifj_abc_cut  QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...   
2  5ig7_abc_cut  EVQLLESGGGLVQPGGSLRLSCAASGFTFSTYAMSWVRQAPGKGLE...   
3  5ig7_abc_cut  QLVLTQSPSASASLGASVRLTCTLSSGHSSYVIAWHQQQSEKGPRY...   
4  5ijk_acx_cut  EVQLVESGGGLVKPGGSLRLSCAASGFTFSNAWFNWVRQAPGKGLE...   

            target  af3_plddt  
0  gliadin_alpha_E  95.404666  
1  gliadin_alpha_E  95.404666  
2  gliadin_alpha_Q  95.599717  
3  gliadin_alpha_Q  95.599717  
4  gliadin_gamma_E  92.439163  


In [16]:
print(f"Dataset shape: {df_interface.shape}")
print(f"Binding distribution:\n{df_interface['binder'].value_counts()}")
print(df_interface.isna().sum().sort_values(ascending=False).to_string())

Dataset shape: (66, 148)
Binding distribution:
binder
0    60
1     6
Name: count, dtype: int64
target_residues                           66
esmfold2_iptm                             66
esmfold2_plddt                            66
af3_rank0_masif                           66
esmfold2_ptm                              66
esmfold2_iplddt                           66
esmfold2_ipae                             66
esmfold2_pae                              66
cf_rank0_masif                            64
esmfold_iplddt                            64
esmfold_ipae                              64
notes                                     60
KD[M]                                     60
chai_constrained_pdockq2                   4
chai_constrained_lis                       4
chai_constrained_ipsae                     4
chai_constrained_pdockq                    4
sample                                     0
fab                                        0
source                                     0
af3_

In [17]:
df_interface.info()

<class 'pandas.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Columns: 148 entries, source to esmfold_mpnn_nll_b
dtypes: bool(1), float64(130), int64(1), str(16)
memory usage: 161.7 KB


In [18]:
print(df_interface.columns.tolist())

['source', 'type', 'sample', 'fab', 'interface', 'vh/vl', 'target', 'target_residues', 'target_sequence', 'binder', 'KD[M]', 'notes', 'binding', 'af3_pae', 'af3_plddt', 'af3_interface_dG', 'af3_interface_dG_SASA_ratio', 'af3_ipae', 'af3_iplddt', 'af3_ipsae', 'af3_pdockq', 'af3_pdockq2', 'af3_lis', 'af3_iptm', 'af3_ptm', 'af3_pyros_binder_score', 'af3_rank', 'af3_ranking_score', 'af3_rank0_masif', 'cf_af3_rmsd', 'cf_pae', 'cf_plddt', 'cf_interface_dG', 'cf_interface_dG_SASA_ratio', 'cf_ipae', 'cf_iplddt', 'cf_ipsae', 'cf_pdockq', 'cf_pdockq2', 'cf_lis', 'cf_iptm', 'cf_ptm', 'cf_pyros_binder_score', 'cf_rank', 'cf_rank0_masif', 'pred_ilddt', 'pred_lddt', 'esmfold_ipae', 'esmfold_iplddt', 'esmfold_plddt', 'esmfold_pae', 'esmfold_ptm', 'esmfold2_ipae', 'esmfold2_iplddt', 'esmfold2_plddt', 'esmfold2_pae', 'esmfold2_ptm', 'esmfold2_iptm', 'chai_unconstrained_score', 'chai_unconstrained_model_path', 'chai_constrained_score', 'chai_constrained_model_path', 'chai_unconstrained_aggregate_score',

In [19]:
df_interface.drop(columns=['fab','vh/vl', 'target', 'target_residues', 'target_sequence', 'notes',], inplace=True)
df_interface.head()

,source,type,sample,interface,binder,KD[M],binding,af3_pae,af3_plddt,af3_interface_dG,...,chai_unconstrained_mpnn_nll_a,chai_unconstrained_mpnn_nll_b,esmfold_esm3dg_dg,esmfold_esm3dg_dg_a,esmfold_esm3dg_dg_b,esmfold_esm3dg_dg_std,esmfold_mpnn_confidence,esmfold_mpnn_nll,esmfold_mpnn_nll_a,esmfold_mpnn_nll_b
0,snir_ab,original,5ifj_abc_cut,"A,C",1,0.000010,False,2.593883,95.404666,16.50,...,1.067877,1.463277,3.275588,4.622507,1.928668,0.272223,0.149689,1.899199,1.302920,2.495477
1,snir_ab,original,5ifj_abc_cut,"B,C",1,0.000010,False,2.593883,95.404666,12.88,...,1.252996,1.463277,3.742723,5.556777,1.928668,0.223455,0.148120,1.909733,1.323989,2.495477
2,snir_ab,original,5ig7_abc_cut,"A,C",1,0.000100,False,2.572969,95.599717,10.97,...,1.183717,2.001695,3.147559,4.633100,1.662018,0.261472,0.143766,1.939571,1.308056,2.571085
3,snir_ab,original,5ig7_abc_cut,"B,C",1,0.000100,False,2.572969,95.599717,5.13,...,1.312539,2.001695,3.601259,5.540500,1.662018,0.216745,0.142654,1.947334,1.323583,2.571085
4,snir_ab,original,5ijk_acx_cut,"A,C",1,0.000001,False,3.486349,92.439163,33.54,...,1.070151,2.297401,3.314784,5.422896,1.206672,0.195615,0.153344,1.875071,1.116188,2.633954


In [20]:
# every sample has two interfaces (A,C and B,C), the the two interfaces should be merged per sample, for of the numerical values mean should be calculated and the "interface" column should be dropped
# chai output

# Group by sample and calculate mean for numeric columns
df_merged = df_interface.groupby('sample').agg({
    col: 'mean' for col in df_interface.select_dtypes(include=['number']).columns
}).reset_index()

# For non-numeric columns (like 'binder'), take the first value (same for both interfaces)
non_numeric_cols = ['source','binding', 'binder', 'type', 'interface', 'chai_unconstrained_per_chain_ptm', 'chai_unconstrained_per_chain_pair_iptm', 'chai_constrained_per_chain_ptm', 'chai_constrained_per_chain_pair_iptm','chai_unconstrained_model_path', 'chai_constrained_model_path', 'boltz_free_model_path', 'boltz_template_model_path']  # add any other non-numeric columns you want to keep
for col in non_numeric_cols:
    if col in df_interface.columns:
        df_merged[col] = df_interface.groupby('sample')[col].first().values

# Reorder columns to match original structure
df_merged = df_merged[['sample'] + [c for c in df_merged.columns if c != 'sample']]

print(f"Original shape: {df_interface.shape}")
print(f"Merged shape: {df_merged.shape}")
print(df_merged.head())

# Save the merged dataframe
df_merged.to_csv(OUTPUT_DIR / 'df_merged_interfaces.csv', index=False)

Original shape: (66, 142)
Merged shape: (33, 142)
                 sample  binder    KD[M]   af3_pae  af3_plddt  \
0          5ifj_abc_cut       1  0.00001  2.593883  95.404666   
1  5ifj_abc_cut_A_D111A       0      NaN  2.815847  95.282854   
2  5ifj_abc_cut_A_K110A       0      NaN  2.790728  94.871984   
3   5ifj_abc_cut_A_S64A       0      NaN  2.539446  95.613237   
4   5ifj_abc_cut_A_W52A       0      NaN  3.005329  94.814333   

   af3_interface_dG  af3_interface_dG_SASA_ratio  af3_ipae  af3_iplddt  \
0            14.690                        1.855  1.993750   95.036725   
1            13.225                        1.745  2.100000   94.730775   
2            19.975                        2.845  2.275417   92.998114   
3            14.595                        1.810  2.072188   94.813927   
4            46.435                        5.940  2.303125   91.482646   

   af3_ipsae  ...          type  interface  \
0   0.788184  ...      original        A,C   
1   0.765295  ...  ala

/tmp/ipykernel_6197/4180807994.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  }).reset_index()
/tmp/ipykernel_6197/4180807994.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged[col] = df_interface.groupby('sample')[col].first().values
/tmp/ipykernel_6197/4180807994.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented fram

In [21]:
print(df_merged.columns.tolist())

['sample', 'binder', 'KD[M]', 'af3_pae', 'af3_plddt', 'af3_interface_dG', 'af3_interface_dG_SASA_ratio', 'af3_ipae', 'af3_iplddt', 'af3_ipsae', 'af3_pdockq', 'af3_pdockq2', 'af3_lis', 'af3_iptm', 'af3_ptm', 'af3_pyros_binder_score', 'af3_rank', 'af3_ranking_score', 'af3_rank0_masif', 'cf_af3_rmsd', 'cf_pae', 'cf_plddt', 'cf_interface_dG', 'cf_interface_dG_SASA_ratio', 'cf_ipae', 'cf_iplddt', 'cf_ipsae', 'cf_pdockq', 'cf_pdockq2', 'cf_lis', 'cf_iptm', 'cf_ptm', 'cf_pyros_binder_score', 'cf_rank', 'cf_rank0_masif', 'pred_ilddt', 'pred_lddt', 'esmfold_ipae', 'esmfold_iplddt', 'esmfold_plddt', 'esmfold_pae', 'esmfold_ptm', 'esmfold2_ipae', 'esmfold2_iplddt', 'esmfold2_plddt', 'esmfold2_pae', 'esmfold2_ptm', 'esmfold2_iptm', 'chai_unconstrained_score', 'chai_constrained_score', 'chai_unconstrained_aggregate_score', 'chai_unconstrained_ptm', 'chai_unconstrained_iptm', 'chai_unconstrained_interface_iptm', 'chai_unconstrained_ipsae', 'chai_unconstrained_pdockq', 'chai_unconstrained_pdockq2', '

In [22]:
cols_to_drop=(['binding', 'interface', 'af3_rank', 'af3_ranking_score', 'af3_rank0_masif', 'cf_rank', 'cf_rank0_masif', 'chai_unconstrained_model_path', 'chai_constrained_model_path', 'boltz_free_model_path', 'boltz_template_model_path'])

scores_df = df_merged.drop(columns=cols_to_drop)

# vllt drop 'KD[M]', 'EC50[M]' too?
scores_df.head()


,sample,binder,KD[M],af3_pae,af3_plddt,af3_interface_dG,af3_interface_dG_SASA_ratio,af3_ipae,af3_iplddt,af3_ipsae,...,esmfold_mpnn_confidence,esmfold_mpnn_nll,esmfold_mpnn_nll_a,esmfold_mpnn_nll_b,source,type,chai_unconstrained_per_chain_ptm,chai_unconstrained_per_chain_pair_iptm,chai_constrained_per_chain_ptm,chai_constrained_per_chain_pair_iptm
0,5ifj_abc_cut,1,0.00001,2.593883,95.404666,14.690,1.855,1.993750,95.036725,0.788184,...,0.148904,1.904466,1.313454,2.495477,snir_ab,original,"[[0.9473640322685242, 0.9504183530807495, 0.17...","[[[0.9473640322685242, 0.8909199237823486, 0.0...","[[0.9454320669174194, 0.9492036700248718, 0.17...","[[[0.9454320669174194, 0.8899917602539062, 0.0..."
1,5ifj_abc_cut_A_D111A,0,NaN,2.815847,95.282854,13.225,1.745,2.100000,94.730775,0.765295,...,0.157618,1.847591,1.331100,2.364081,snir_ab,alanine_scan,"[[0.9472975134849548, 0.9492976665496826, 0.17...","[[[0.9472975134849548, 0.8904723525047302, 0.0...","[[0.9454193711280823, 0.9488046169281006, 0.17...","[[[0.9454193711280823, 0.8907997608184814, 0.0..."
2,5ifj_abc_cut_A_K110A,0,NaN,2.790728,94.871984,19.975,2.845,2.275417,92.998114,0.769552,...,0.155875,1.858766,1.307114,2.410418,snir_ab,alanine_scan,"[[0.9461949467658997, 0.9488812685012817, 0.16...","[[[0.9461949467658997, 0.8897757530212402, 0.0...","[[0.9449247717857361, 0.9489190578460693, 0.17...","[[[0.9449247717857361, 0.889130175113678, 0.07..."
3,5ifj_abc_cut_A_S64A,0,NaN,2.539446,95.613237,14.595,1.810,2.072188,94.813927,0.778962,...,0.149138,1.903001,1.292423,2.513579,snir_ab,alanine_scan,"[[0.9445436000823975, 0.9495869874954224, 0.17...","[[[0.9445436000823975, 0.8924098014831543, 0.0...","[[0.945385217666626, 0.948758602142334, 0.1724...","[[[0.945385217666626, 0.8885297775268555, 0.07..."
4,5ifj_abc_cut_A_W52A,0,NaN,3.005329,94.814333,46.435,5.940,2.303125,91.482646,0.695928,...,0.143529,1.941261,1.309289,2.573233,snir_ab,alanine_scan,"[[0.9452440142631531, 0.9471760392189026, 0.16...","[[[0.9452440142631531, 0.8792759776115417, 0.0...","[[0.9452540874481201, 0.9489129781723022, 0.17...","[[[0.9452540874481201, 0.8904703855514526, 0.0..."


In [23]:
scores_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 33 entries, 0 to 32
Columns: 131 entries, sample to chai_constrained_per_chain_pair_iptm
dtypes: float64(123), int64(1), str(7)
memory usage: 51.4 KB


In [24]:
# Save the merged result
scores_df.to_csv(OUTPUT_DIR /'merged_snir.csv', index=False)